# Phishing URL Detection — PhiUSIIL Lab (bias-aware, Flask-ready)

**Goal:** Compare **Logistic Regression, Random Forest, XGBoost, and Soft Voting** using raw PhiUSIIL URLs; perform data audit, training-only feature selection, ROC/PR evaluation, SHAP explanation, and export the exact feature extractor + selected model for Flask and frontend use.

**Ground truth:** `0 = PHISHING`, `1 = LEGITIMATE`. The model's *phishing score* must always use `model.classes_ == 0`.

**Known limitation (from previous experiments):** PhiUSIIL legitimate URLs overwhelmingly use `www.` and contain no paths/queries. Earlier full-URL models marked 100% of synthetic benign path/query variations as phishing. **This notebook removes leading `www.` and ignores URL scheme/path/query entirely**. That removes those specific shortcuts but also loses path-based attack information. **No model trained only on this corpus can be guaranteed reliable on real-world URLs.** A separate benign + phishing external evaluation is required before production deployment.

The final application returns **LIKELY PHISHING / INSPECT / LIKELY LEGITIMATE** rather than auto-blocking URLs. The URL is never visited.

## Phase 0 — Install packages (run this cell first)
Colab needs internet access to fetch UCI dataset ID 967. Installation output is deliberately quiet.

In [ ]:
%pip -q install "ucimlrepo>=0.0.7" "tldextract>=5,<6" "xgboost>=2,<4" "shap>=0.45,<1" "flask>=3,<4" "seaborn>=0.13,<1" "joblib>=1.3,<2"

In [ ]:
from pathlib import Path
import os, sys
PROJECT_DIR = Path.cwd() if Path.cwd().name == 'phishing_url_lab' else (Path.cwd() / 'phishing_url_lab').resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Working directory:", PROJECT_DIR)

## Phase 1 — Write shared, versioned URL feature extractor
This **exact file** is also used by Flask. Any change here requires retraining the model. We do **not** use UCI precomputed feature columns.

In [ ]:
%%writefile features.py
"""Versioned, URL-only features used identically by training and Flask.

Intentional safeguards for a biased training corpus:
- Remove a leading www. before computing hostname statistics.
- Ignore URL scheme, paths, queries, and fragments entirely.
- Do not use PhiUSIIL's precomputed, label-correlated URLSimilarityIndex.

Limitation: hostname-only features cannot recognize attacks that live solely in
paths on otherwise legitimate domains. No website is visited by this module.
"""
from __future__ import annotations

import ipaddress
import math
import re
from collections import Counter
from urllib.parse import urlsplit

import tldextract

FEATURE_VERSION = "hostname_safe_v1"
FEATURE_NAMES = [
    "domain_label_length", "domain_digit_count", "domain_digit_ratio",
    "domain_hyphen_count", "domain_entropy", "domain_vowel_ratio",
    "tld_length", "host_length", "host_digit_count", "host_digit_ratio",
    "host_hyphen_count", "host_alpha_ratio", "host_entropy",
    "host_longest_digit_run", "host_longest_letter_run", "host_has_punycode",
    "host_is_ip",
]

# Uses packaged Public Suffix List snapshot. No background network requests.
_PSL = tldextract.TLDExtract(suffix_list_urls=(), fallback_to_snapshot=True)
_SCHEME = re.compile(r"^[A-Za-z][A-Za-z0-9+.-]*://")


def _url_parts(raw_url: str):
    if not isinstance(raw_url, str) or not raw_url.strip():
        raise ValueError("Provide a nonempty URL string.")
    raw = raw_url.strip()
    if len(raw) > 8192:
        raise ValueError("URL is too long (maximum: 8192 characters).")
    if any(ch.isspace() for ch in raw):
        raise ValueError("URLs containing unescaped whitespace are not supported.")
    if raw.startswith("//"):
        raw = "http:" + raw
    elif not _SCHEME.match(raw):
        # Reject strings like javascript:alert(1), mailto:, or data:.
        if re.match(r"^[A-Za-z][A-Za-z0-9+.-]*:", raw):
            raise ValueError("Only HTTP and HTTPS URLs are supported.")
        raw = "http://" + raw
    try:
        parsed = urlsplit(raw)
        if parsed.scheme.lower() not in {"http", "https"}:
            raise ValueError("Only HTTP and HTTPS URLs are supported.")
        hostname = (parsed.hostname or "").lower().rstrip(".")
        if not hostname or hostname.startswith("."):
            raise ValueError("URL is missing a valid hostname.")
    except (ValueError, UnicodeError) as exc:
        raise ValueError("Invalid URL: " + str(exc)) from exc
    # Strip www to prevent a shortcut inherited from PhiUSIIL.
    if hostname.startswith("www."):
        hostname = hostname[4:]
    return hostname


def registered_domain_from_url(url: str) -> str:
    host = _url_parts(url)
    try:
        ipaddress.ip_address(host)
        return host
    except ValueError:
        pass
    parts = _PSL(host)
    if parts.domain and parts.suffix:
        return f"{parts.domain}.{parts.suffix}"
    return host


def _entropy(s: str) -> float:
    if not s:
        return 0.0
    counts = Counter(s)
    n = len(s)
    return float(-sum((count / n) * math.log2(count / n)
                      for count in counts.values()))


def _longest_run(pattern: str, text: str) -> int:
    return max((len(m.group()) for m in re.finditer(pattern, text)), default=0)


def extract_lexical_features(url: str) -> dict[str, float]:
    """Return deterministic numeric features; never fetches the supplied URL."""
    host = _url_parts(url)
    try:
        ipaddress.ip_address(host)
        is_ip = 1
    except ValueError:
        is_ip = 0
    ext = _PSL(host) if not is_ip else None
    domain = ext.domain if ext and ext.domain else host
    suffix = ext.suffix if ext else ""

    def digits(s):
        return sum(ch.isdigit() for ch in s)
    def letters(s):
        return sum(ch.isalpha() for ch in s)

    host_len = max(len(host), 1)
    dom_len = max(len(domain), 1)
    host_digits = digits(host)
    domain_digits = digits(domain)
    result = {
        "domain_label_length": len(domain),
        "domain_digit_count": domain_digits,
        "domain_digit_ratio": domain_digits / dom_len,
        "domain_hyphen_count": domain.count("-"),
        "domain_entropy": _entropy(domain),
        "domain_vowel_ratio": sum(c in "aeiou" for c in domain) / dom_len,
        "tld_length": len(suffix),
        "host_length": len(host),
        "host_digit_count": host_digits,
        "host_digit_ratio": host_digits / host_len,
        "host_hyphen_count": host.count("-"),
        "host_alpha_ratio": letters(host) / host_len,
        "host_entropy": _entropy(host),
        "host_longest_digit_run": _longest_run(r"[0-9]+", host),
        "host_longest_letter_run": _longest_run(r"[a-z]+", host),
        "host_has_punycode": int("xn--" in host),
        "host_is_ip": is_ip,
    }
    return {name: float(result[name]) for name in FEATURE_NAMES}

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from ucimlrepo import fetch_ucirepo
from urllib.parse import urlsplit, urlunsplit
from features import FEATURE_NAMES, FEATURE_VERSION, extract_lexical_features, registered_domain_from_url

SEED=42
np.random.seed(SEED)
print("Feature version:", FEATURE_VERSION)
print("Candidate hostname features:", len(FEATURE_NAMES))
print(FEATURE_NAMES)

# Smoke tests — path/query and leading www must not alter features
assert extract_lexical_features('https://www.example.org/') == extract_lexical_features('http://example.org/a?q=42')
assert extract_lexical_features('https://portal.example.org/a') == extract_lexical_features('http://portal.example.org/?z=x')
print("Extractor consistency tests passed.")

## Phase 2 — Fetch PhiUSIIL directly from UCI and audit labels
We retain only the original `URL` and its target label, **not** PhiUSIIL's precomputed 54 features.

In [ ]:
ds = fetch_ucirepo(id=967)
raw_url = ds.data.features['URL'].astype('string')
raw_y = pd.to_numeric(ds.data.targets.iloc[:, 0], errors='coerce')
raw_df = pd.DataFrame({'URL': raw_url, 'label': raw_y})
print("UCI raw rows:", len(raw_df))
print("Before cleaning labels:")
print(raw_df['label'].value_counts(dropna=False))

# Clean labels and URLs, check conflicts before deduplication
raw_df = raw_df.dropna(subset=['URL', 'label']).copy()
raw_df['URL'] = raw_df['URL'].str.strip()
raw_df = raw_df[raw_df['URL'].str.len().between(1, 8192)].copy()
assert set(raw_df['label'].unique()) <= {0, 1}, 'Expected UCI labels 0 and 1.'
raw_df['label'] = raw_df['label'].astype(int)

conflicts = raw_df.groupby('URL')['label'].nunique().gt(1).sum()
print('URLs with conflicting labels:', int(conflicts))
if conflicts:
    raise ValueError('Resolve URLs with conflicting labels before continuing.')

raw_df = raw_df.drop_duplicates(subset=['URL'], keep='first').reset_index(drop=True)

# Filter malformed HTTP(S) URLs (feature extraction rejects bad strings)
valid=[]
for u in raw_df['URL']:
    try:
        extract_lexical_features(u)
        valid.append(True)
    except ValueError:
        valid.append(False)
print('Rejected invalid URLs:', int(len(valid) - sum(valid)))
df=raw_df.loc[valid].reset_index(drop=True)
print('After cleaning:', df.shape)
print('Class distribution (0 phishing, 1 legitimate):')
print(df['label'].value_counts())
assert df['label'].nunique() == 2

### Inspect sampling bias before learning
This audit is **not** a feature-selection step. It warns about skewed collection patterns. Percentages near 0%/100% signal strong shortcuts.

In [ ]:
def url_flags(raw):
    u = str(raw).strip()
    if not u.startswith(('http://', 'https://')):
        u='http://'+u
    p = urlsplit(u)
    return (int(p.path not in ('', '/')), int(bool(p.query)), int((p.hostname or '').lower().startswith('www.')))
flags=pd.DataFrame([url_flags(u) for u in df['URL']], columns=['HasPath', 'HasQuery', 'HasWWW'])
flags['label'] = df['label'].values
print('Percent of each class having each URL structure:')
display(flags.groupby('label')[['HasPath','HasQuery','HasWWW']].mean().mul(100).round(2))
print('WARNING: structural label correlations in this dataset still exist even if those raw features are excluded.')

## Phase 3 — Leakage-aware train / validation / test groups
Split registered domains (eTLD+1) **before any learned feature selection or model fitting**. The test split is reserved for one final evaluation only.

- Train: approximately 70%
- Validation: approximately 15% (model/threshold selection)
- Holdout test: approximately 15% (used once after selection)

Whole-domain grouping can cause class proportions to differ; this is not a stratified random split.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df['URL'].map(registered_domain_from_url)
X_urls = df['URL']
y = df['label']

split_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
devval_idx, test_idx = next(split_test.split(X_urls, y, groups=groups))

# 0.17647 * remaining 85% ~= 15% of original
split_val = GroupShuffleSplit(n_splits=1, test_size=0.1764706, random_state=SEED+1)
dev_local, val_local = next(split_val.split(
    X_urls.iloc[devval_idx], y.iloc[devval_idx],
    groups=groups.iloc[devval_idx]))

dev_idx=devval_idx[dev_local]
val_idx=devval_idx[val_local]

def make_partition(indices):
    return (X_urls.iloc[indices].reset_index(drop=True),
            y.iloc[indices].reset_index(drop=True),
            groups.iloc[indices].reset_index(drop=True))

dev_urls, y_dev, dev_groups=make_partition(dev_idx)
val_urls, y_val, val_groups=make_partition(val_idx)
test_urls, y_test, test_groups=make_partition(test_idx)

assert not (set(dev_groups) & set(val_groups))
assert not (set(dev_groups) & set(test_groups))
assert not (set(val_groups) & set(test_groups))
for title, target in [('Development', y_dev), ('Validation', y_val), ('Holdout', y_test)]:
    print(title, 'rows:', len(target), '| phishing:', round((target == 0).mean()*100,2),'%')
    assert target.nunique()==2, f'{title} must contain both classes'
print('No shared registered domains among development, validation and test.')

## Phase 4 — Deterministic feature engineering
No webpage fetches. We intentionally **ignore full URL path, query, scheme, www**, and `URLSimilarityIndex` to reduce the strongest identified biases.

In [ ]:
def features_frame(urls):
    return pd.DataFrame([extract_lexical_features(u) for u in urls], columns=FEATURE_NAMES)

X_dev_all = features_frame(dev_urls)
X_val_all = features_frame(val_urls)
X_test_all = features_frame(test_urls)
assert X_dev_all.columns.tolist() == FEATURE_NAMES
assert X_dev_all.notna().all().all() and X_val_all.notna().all().all() and X_test_all.notna().all().all()
print('Training:', X_dev_all.shape, 'Validation:', X_val_all.shape, 'Test:', X_test_all.shape)
display(X_dev_all.head())

## Phase 5 — Feature selection using development training **only**
Train-only mutual information ranking; prune pairs with absolute **Spearman correlation ≥ 0.97**; keep at most **12 ranked, nonredundant** features. The choice of 12 is a simple lab complexity constraint, not an optimal guarantee. No test labels are used in selection.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

sample_n=min(30000,len(X_dev_all))
mi_sample=X_dev_all.sample(n=sample_n, random_state=SEED)
mi_y=y_dev.loc[mi_sample.index]
discrete=[name in {'host_has_punycode','host_is_ip'} for name in FEATURE_NAMES]
mi_scores=mutual_info_classif(mi_sample, mi_y, discrete_features=discrete, random_state=SEED)
ranking=pd.Series(mi_scores,index=FEATURE_NAMES,name='MutualInformation').sort_values(ascending=False)
correlations=X_dev_all.corr(method='spearman').abs().fillna(0)
selected=[]
for name in ranking.index:
    if X_dev_all[name].nunique()<2:
        continue
    if all(correlations.loc[name,other]<0.97 for other in selected):
        selected.append(name)
    if len(selected)==12:
        break
assert selected, 'No useful features selected'
print('Feature ranking:')
display(ranking.to_frame())
print('Selected features:',selected)

plt.figure(figsize=(9,6))
ranking.sort_values().plot.barh()
plt.title('Mutual information — development data only')
plt.xlabel('Estimated information (nats)')
plt.tight_layout()
plt.show()

X_dev=X_dev_all[selected].copy()
X_val=X_val_all[selected].copy()
X_test=X_test_all[selected].copy()

## Phase 6 — Train four comparable classifiers

Models: standardized Logistic Regression, Random Forest, XGBoost, and soft voting of the three. No automatic resampling or SMOTE. Training can take several minutes on CPU because Soft Voting refits its base estimators.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier


def logistic_model():
    return make_pipeline(StandardScaler(), LogisticRegression(
        max_iter=1500, class_weight='balanced', random_state=SEED))

def forest_model():
    return RandomForestClassifier(
        n_estimators=140, max_depth=14, min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)

def xgb_model():
    return XGBClassifier(
        n_estimators=180, max_depth=5, learning_rate=0.06,
        subsample=0.85, colsample_bytree=0.9,
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', n_jobs=2, random_state=SEED)

models = {
    'Logistic Regression': logistic_model(),
    'Random Forest': forest_model(),
    'XGBoost': xgb_model(),
    'Soft Voting': VotingClassifier(
        estimators=[('lr',logistic_model()),('rf',forest_model()),('xgb',xgb_model())],
        voting='soft', n_jobs=1)
}

from time import perf_counter
for name,model in models.items():
    started=perf_counter()
    model.fit(X_dev,y_dev)
    print(f'{name}: trained in {perf_counter()-started:.1f} seconds | classes={model.classes_.tolist()}')

## Phase 7 — Threshold optimization and validation performance
Phishing is class **0**, so `phishing_score = predict_proba(...)[index of class 0]`. We select a decision threshold **only using validation**, maximizing phishing F1 **subject to a validation false-alarm-rate ceiling of 5%**. If no threshold meets the constraint, the code explicitly reports a fallback. This is not a real-world false-alarm guarantee.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
  precision_score, recall_score, f1_score, confusion_matrix,
  roc_auc_score, average_precision_score, brier_score_loss)

TARGET_FALSE_ALARM_RATE=0.05

def phishing_scores(model, data):
    return model.predict_proba(data)[:,list(model.classes_).index(0)]

def score_metrics(actual_labels, prob, threshold):
    pred=np.where(prob>=threshold, 0, 1)
    cm=confusion_matrix(actual_labels,pred,labels=[0,1])
    detected,missed=map(int,cm[0]); false_alarm,correct_legit=map(int,cm[1])
    return dict(
        Accuracy=accuracy_score(actual_labels,pred),
        BalancedAccuracy=balanced_accuracy_score(actual_labels,pred),
        PhishingPrecision=precision_score(actual_labels,pred,pos_label=0,zero_division=0),
        PhishingRecall=recall_score(actual_labels,pred,pos_label=0,zero_division=0),
        PhishingF1=f1_score(actual_labels,pred,pos_label=0,zero_division=0),
        FalseAlarmRate=false_alarm/max(1,false_alarm+correct_legit),
        MissedPhishing=missed, FalseAlarms=false_alarm,
    )

val_prob={name:phishing_scores(model,X_val) for name,model in models.items()}
selection_rows=[]
for name,prob in val_prob.items():
    for threshold in np.round(np.arange(.05,.951,.025),3):
        row=score_metrics(y_val,prob,float(threshold))
        selection_rows.append({'Model':name,'Threshold':float(threshold),**row})
selection_table=pd.DataFrame(selection_rows)
feasible=selection_table[selection_table.FalseAlarmRate<=TARGET_FALSE_ALARM_RATE]
if feasible.empty:
    print('WARNING: No evaluated threshold satisfies FPR <= 5%. Using highest F1 as a fallback.')
    candidates=selection_table
else:
    candidates=feasible
winner_row=candidates.sort_values(['PhishingF1','PhishingRecall'],ascending=False).iloc[0]
winner_name=str(winner_row['Model'])
winner_threshold=float(winner_row['Threshold'])
winner=models[winner_name]

# Table: each model at its best feasible validation operating point
per_model=[]
for name,rows in selection_table.groupby('Model'):
    allowed=rows[rows.FalseAlarmRate<=TARGET_FALSE_ALARM_RATE]
    best=(allowed if not allowed.empty else rows).sort_values('PhishingF1',ascending=False).iloc[0]
    positive=(y_val==0).astype(int)
    per_model.append({**best.to_dict(),
       'ROC_AUC':roc_auc_score(positive,val_prob[name]),
       'PR_AUC':average_precision_score(positive,val_prob[name]),
       'BrierScore':brier_score_loss(positive,val_prob[name])})
validation_results=pd.DataFrame(per_model).sort_values('PhishingF1',ascending=False)
print('Validation model comparison (threshold tuned on validation only):')
display(validation_results.round(4))
print(f'CHAMPION (provisional): {winner_name} at score threshold {winner_threshold:.3f}')

### Validation ROC and precision–recall curves
Curves use class **0 = phishing as the positive class**. PR-AUC can be strongly influenced by class prevalence. Curves on PhiUSIIL validation are *not* evidence of deployment safety.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
fig,axes=plt.subplots(1,2,figsize=(13,5))
pos=(y_val==0).astype(int)
for name,prob in val_prob.items():
    fpr,tpr,_=roc_curve(pos,prob)
    p,r,_=precision_recall_curve(pos,prob)
    axes[0].plot(fpr,tpr,label=f'{name} (AUC={roc_auc_score(pos,prob):.3f})')
    axes[1].plot(r,p,label=f'{name} (AP={average_precision_score(pos,prob):.3f})')
axes[0].plot([0,1],[0,1],'--',color='gray',alpha=.6)
axes[0].set(title='Validation ROC',xlabel='False Positive Rate',ylabel='True Positive Rate')
axes[1].set(title='Validation Precision–Recall',xlabel='Phishing Recall',ylabel='Phishing Precision')
for ax in axes:
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Phase 8 — Bias stress test on validation examples (does not retrain)
For a URL-only classifier we deliberately ignore path/query and strip leading `www.`. Therefore a **synthetic change** to those components should not alter a prediction. This checks for specific shortcut resistance, but synthetic modified URLs are not confirmed live or legitimate sites.

In [ ]:
def variation(url,mode):
    s=str(url)
    if '://' not in s: s='http://'+s
    p=urlsplit(s)
    if mode=='add_path':
        return urlunsplit((p.scheme,p.netloc,'/docs/getting-started',p.query,p.fragment))
    if mode=='add_query':
        return urlunsplit((p.scheme,p.netloc,p.path,'q=sample&page=2',p.fragment))
    if mode=='remove_www':
        netloc=p.netloc[4:] if p.netloc.lower().startswith('www.') else p.netloc
        return urlunsplit((p.scheme,netloc,p.path,p.query,p.fragment))
    return url

benign_val_urls=val_urls.loc[y_val==1].sample(n=min(500,len(y_val[y_val==1])),random_state=SEED)
base_scores=phishing_scores(winner,features_frame(benign_val_urls)[selected])
stress=[]
for mode in ['original','add_path','add_query','remove_www']:
    scores=phishing_scores(winner,features_frame(benign_val_urls.map(lambda u:variation(u,mode)))[selected])
    stress.append({'Scenario':mode, 'Count':len(scores),
                   'FlaggedPercent':float((scores>=winner_threshold).mean()*100),
                   'MedianPhishingScore':float(np.median(scores)),
                   'MaxAbsoluteScoreChange':float(np.max(np.abs(scores-base_scores)))})
display(pd.DataFrame(stress).round(6))
assert max(row['MaxAbsoluteScoreChange'] for row in stress)<1e-9, 'Prediction invariance regression: feature policy changed!'
print('Passed: scheme/www/path/query transformations do not change model output.')

## Phase 9 — One-time evaluation on untouched PhiUSIIL test split
This tests the chosen threshold and model exactly once after using the validation split for selection. It does **not** measure distribution shift to independent modern legitimate/phishing data.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
holdout_prob=phishing_scores(winner,X_test)
final_metrics=score_metrics(y_test,holdout_prob,winner_threshold)
final_metrics['ROC_AUC']=roc_auc_score((y_test==0).astype(int),holdout_prob)
final_metrics['PR_AUC']=average_precision_score((y_test==0).astype(int),holdout_prob)
print('FINAL HOLDOUT RESULT (PhiUSIIL; not external generalization):')
display(pd.DataFrame([final_metrics]).round(4))
final_pred=np.where(holdout_prob>=winner_threshold,0,1)
ConfusionMatrixDisplay.from_predictions(y_test,final_pred,labels=[0,1],
 display_labels=['Phishing (0)','Legitimate (1)'],cmap='Blues')
plt.title('Untouched domain-disjoint holdout confusion matrix')
plt.show()

## Phase 10 — SHAP explanations for the selected model
One **model-agnostic SHAP permutation explainer** works with Logistic Regression, Random Forest, XGBoost and Soft Voting. It explains the **model-reported phishing score** (class 0), NOT causation. For performance, use a small background sample and explain a single URL.

In [ ]:
import shap

shap_background=X_dev.sample(n=min(25,len(X_dev)),random_state=SEED).to_numpy(dtype=float)

def explain_phish_probability(a):
    a=np.asarray(a,dtype=float)
    if a.ndim==1: a=a.reshape(1,-1)
    return phishing_scores(winner,pd.DataFrame(a,columns=selected))

shap_explainer=shap.Explainer(
    explain_phish_probability, shap_background,
    algorithm='permutation', feature_names=selected
)

# Explain a URL sampled from the validation set; no test-set explanations during tuning.
example_idx=0
example=X_val.iloc[[example_idx]]
shap_values=shap_explainer(example.to_numpy(dtype=float),max_evals=2*len(selected)+1)
print('Example URL:',val_urls.iloc[example_idx])
print('True label:',int(y_val.iloc[example_idx]),'| Score phishing:',float(phishing_scores(winner,example)[0]))
shap.plots.waterfall(shap_values[0],max_display=12,show=True)

## Phase 11 — Optional: **fresh independent** labeled URL evaluation
Supply a *new* CSV `external_urls.csv` with columns `URL,label` (0 phishing, 1 legitimate). This file is never used for fitting or threshold selection. **Do not use the old OpenPhish 145 URLs for an untouched final test** because previous experiments used them to choose modeling decisions. Verify labels independently; popular domains do not verify individual benign URLs.

In [ ]:
external_path=Path('external_urls.csv')
if external_path.exists():
    ext=pd.read_csv(external_path)
    if not {'URL','label'}.issubset(ext.columns):
        raise ValueError('external_urls.csv requires URL and label columns')
    ext=ext.dropna(subset=['URL','label']).copy()
    ext['label']=pd.to_numeric(ext['label'],errors='coerce')
    ext=ext.dropna(subset=['label'])
    ext=ext[ext['label'].isin([0,1])].drop_duplicates(subset=['URL'])
    # Exclude known registered domains to reduce overlap
    training_domains=set(dev_groups)|set(val_groups)|set(test_groups)
    ext=ext[~ext['URL'].map(registered_domain_from_url).isin(training_domains)]
    if len(ext)>0:
        external_X=features_frame(ext['URL'])[selected]
        external_y=ext['label'].astype(int).reset_index(drop=True)
        external_prob=phishing_scores(winner,external_X)
        print('External domain-disjoint URLs:',len(ext),'| class distribution:',external_y.value_counts().to_dict())
        display(pd.DataFrame([score_metrics(external_y,external_prob,winner_threshold)]).round(4))
        if external_y.nunique()==2:
            print('External ROC AUC:',roc_auc_score((external_y==0).astype(int),external_prob))
            print('External AP:',average_precision_score((external_y==0).astype(int),external_prob))
        else:
            print('Only one class: precision and general false-alarm behavior cannot be fully assessed.')
    else:
        print('No usable domain-disjoint examples in external_urls.csv')
else:
    print('Optional external_urls.csv not provided; real-world reliability remains UNVERIFIED.')

## Phase 12 — Export selected model and exact preprocessing for Flask
**Export the already chosen model, do not retrain after viewing test results.** The bundle contains model, selected column order, threshold, feature version and SHAP background. The local Flask frontend is in the exported ZIP. Only load this joblib file from a trusted source.

In [ ]:
bundle={
    'model':winner,
    'model_name':winner_name,
    'feature_names':selected,
    'feature_version':FEATURE_VERSION,
    'threshold':winner_threshold,
    'shap_background':shap_background,
    'class_mapping':{0:'Phishing',1:'Legitimate'},
    'warning':'PhiUSIIL-only laboratory model; external reliable benign/phishing evaluation required.'
}
joblib.dump(bundle,'model_bundle.joblib',compress=3)
print('Saved model_bundle.joblib; features:',selected)
print('Bundle size MB:',round(Path('model_bundle.joblib').stat().st_size/1024**2,2))

### Write matching Flask endpoint, frontend, requirements and tests
The notebook recreates the Flask files directly so your Colab export is self-contained.

In [ ]:
%%writefile app.py
"""Local demo only. Reads URL text and never visits the target site.
Run after the notebook writes model_bundle.joblib into this directory:
    python app.py
"""
from __future__ import annotations

from pathlib import Path
import os
import threading

import joblib
import numpy as np
import pandas as pd
from flask import Flask, jsonify, render_template, request

from features import FEATURE_VERSION, extract_lexical_features

APP_DIR = Path(__file__).resolve().parent
BUNDLE_FILE = APP_DIR / "model_bundle.joblib"
if not BUNDLE_FILE.is_file():
    raise RuntimeError("model_bundle.joblib is missing. Run the notebook export cell first.")

# Only load trusted joblib files you trained yourself. joblib/pickle executes code.
bundle = joblib.load(BUNDLE_FILE)
if bundle["feature_version"] != FEATURE_VERSION:
    raise RuntimeError("Feature extractor version does not match trained model!")

MODEL = bundle["model"]
COLUMNS = list(bundle["feature_names"])
THRESHOLD = float(bundle["threshold"])
BACKGROUND = np.asarray(bundle["shap_background"], dtype=float)
INDEX_PHISH = list(MODEL.classes_).index(0)

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 16 * 1024
_shap_lock = threading.Lock()
_explainer = None


def _score_numpy(a):
    arr = np.asarray(a, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    frame = pd.DataFrame(arr, columns=COLUMNS)
    return MODEL.predict_proba(frame)[:, INDEX_PHISH]


def _frame_for_url(url):
    if not isinstance(url, str):
        raise ValueError("URL must be a string.")
    features = extract_lexical_features(url)
    return pd.DataFrame([{key: features[key] for key in COLUMNS}], columns=COLUMNS)


def predict_url(url: str) -> dict:
    features = _frame_for_url(url)
    score = float(MODEL.predict_proba(features)[0, INDEX_PHISH])
    # Abstain around validation-selected threshold; neither side guarantees safety.
    review_band = min(0.08, THRESHOLD / 2, (1 - THRESHOLD) / 2)
    if score >= THRESHOLD + review_band:
        decision = "LIKELY PHISHING"
        label = 0
    elif score <= THRESHOLD - review_band:
        decision = "LIKELY LEGITIMATE"
        label = 1
    else:
        decision = "INSPECT"
        label = None
    return {
        "decision": decision,
        "predicted_label": label,
        "phishing_score": round(score, 6),
        "threshold": round(THRESHOLD, 4),
        "model_name": bundle["model_name"],
        "warning": "URL-only laboratory estimate. Do not treat this as proof that a site is safe.",
    }


def explain_url(url: str) -> list[dict]:
    import shap
    global _explainer
    if _explainer is None:
        with _shap_lock:
            if _explainer is None:
                _explainer = shap.Explainer(
                    _score_numpy, BACKGROUND,
                    algorithm="permutation", feature_names=COLUMNS
                )
    x = _frame_for_url(url)
    # Using one permutation pass to make on-demand explanations practical.
    values = _explainer(x.to_numpy(dtype=float), max_evals=2 * len(COLUMNS) + 1)
    impacts = values.values[0]
    ranking = np.argsort(np.abs(impacts))[::-1][:6]
    return [
        {
            "feature": COLUMNS[int(i)],
            "value": round(float(x.iloc[0, int(i)]), 4),
            "impact_on_phishing_score": round(float(impacts[i]), 5),
            "direction": "raises risk" if impacts[i] > 0 else "lowers risk",
        }
        for i in ranking
    ]


@app.get("/")
def home():
    return render_template("index.html")


@app.post("/api/predict")
def predict():
    body = request.get_json(silent=True) or {}
    try:
        url = body.get("url", "")
        outcome = predict_url(url)
        if body.get("explain") is True:
            outcome["explanation"] = explain_url(url)
        return jsonify(outcome)
    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    except Exception:
        app.logger.exception("Unexpected prediction failure")
        return jsonify({"error": "Prediction failed. Check server logs."}), 500


if __name__ == "__main__":
    # Demo: bind to localhost, disable Flask's debug server.
    app.run(host="127.0.0.1", port=int(os.environ.get("PORT", "5000")), debug=False)

In [ ]:
from pathlib import Path
Path('templates').mkdir(exist_ok=True)
Path('tests').mkdir(exist_ok=True)

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1"/>
<title>URL Safety Lab — Phishing Classifier</title>
<style>
  :root { color-scheme: dark; font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, sans-serif; }
  * {box-sizing:border-box;} body {margin:0; background:#101522; color:#edf2ff; min-height:100vh;}
  .shell {max-width:870px; padding:64px 24px; margin:0 auto;}
  .eyebrow {font-size:12px;letter-spacing:.18em; text-transform:uppercase;color:#91a4c8;font-weight:700;}
  h1 {font-size:clamp(32px,6vw,52px);margin:13px 0;letter-spacing:-.04em;line-height:1.04;}
  .lead{line-height:1.6;color:#aebbd3;max-width:650px;margin-bottom:30px;}
  .panel {border:1px solid #303d57;background:#182235;border-radius:20px;padding:26px;box-shadow:0 14px 40px #070b1480;}
  label {display:block;font-size:14px;font-weight:650;margin-bottom:11px;}
  .field {display:flex; gap:10px;}
  input {min-width:0;flex:1;background:#0e1625;color:#fff;border:1px solid #3c4b66;border-radius:10px;padding:15px 14px;font-size:15px;}
  input:focus {outline:2px solid #7aabff;outline-offset:2px;}
  button {border:0;cursor:pointer;background:#8ab8ff;color:#081324;font-weight:750;border-radius:10px;padding:13px 23px;}
  button:disabled {opacity:.5;cursor:not-allowed;}
  .toggle {display:flex;align-items:center;gap:9px;margin:18px 0 0;color:#c1cee5;font-size:14px;}
  .toggle input {flex:none; width:16px;height:16px;}
  .result {margin-top:22px;display:none;}
  .result.visible {display:block;}
  .top {display:flex;align-items:center;justify-content:space-between;gap:14px;flex-wrap:wrap;}
  .status {font-size:22px;font-weight:750;}
  .number {font-size:28px;font-variant-numeric:tabular-nums;font-weight:800;}
  .danger {color:#ff929e;} .good {color:#85e2bb;} .caution {color:#ffd18a;}
  .track {width:100%;height:11px;background:#2c3750;overflow:hidden;border-radius:20px;margin:18px 0 6px;}
  .fill {height:100%;width:0%;background:#8ab8ff;transition:width .3s ease;}
  .help {font-size:12px;color:#9eaecb;line-height:1.6;}
  .warning {border-top:1px solid #303d57;margin-top:19px;padding-top:16px;color:#b5c3dd;font-size:13px;line-height:1.6;}
  .feature {display:flex;align-items:center;gap:12px;justify-content:space-between;border-bottom:1px solid #2e3a53;padding:10px 0;font-size:13px;}
  .impact {font-weight:800;font-variant-numeric:tabular-nums;}
  h2 {font-size:17px;margin:27px 0 7px;}
  #error {min-height:20px;color:#ff929e;font-size:14px;margin-top:12px;}
  @media(max-width:570px){.shell{padding:36px 16px;}.panel{padding:18px;}.field{flex-direction:column;}button{width:100%;}}
</style>
</head>
<body>
<main class="shell">
  <div class="eyebrow">Academic ML experiment · URL-only</div>
  <h1>Phishing URL analysis</h1>
  <p class="lead">Check the structure of a URL with a trained machine-learning model. Nothing is fetched from the submitted website. Results are estimates, not security guarantees.</p>
  <section class="panel">
    <form id="predict-form">
      <label for="url">URL to analyze</label>
      <div class="field">
        <input id="url" name="url" type="text" inputmode="url" maxlength="8192" autocomplete="off" spellcheck="false" placeholder="https://example.org/login" required />
        <button id="submit" type="submit">Analyze URL</button>
      </div>
      <label class="toggle"><input id="explain" type="checkbox" checked /> Show SHAP feature explanation (slower)</label>
      <div id="error" role="alert" aria-live="polite"></div>
    </form>
    <section id="result" class="result" aria-live="polite">
      <div class="top"><div><div class="eyebrow">Classification</div><div id="status" class="status">—</div></div><div><div class="eyebrow">Phishing model score</div><div id="prob" class="number">—</div></div></div>
      <div class="track"><div id="fill" class="fill"></div></div>
      <div class="help">Score ranges from 0 to 1; it is not a calibrated, real-world probability. INSPECT means the model is uncertain near its validation threshold.</div>
      <div id="explain-list"></div>
      <p class="warning" id="warning"></p>
    </section>
  </section>
</main>
<script>
const form=document.getElementById('predict-form');
form.addEventListener('submit', async (event)=>{
  event.preventDefault();
  const button=document.getElementById('submit');
  const error=document.getElementById('error');
  const result=document.getElementById('result');
  result.classList.remove('visible'); error.textContent=''; button.disabled=true; button.textContent='Analyzing…';
  try {
    const response=await fetch('/api/predict',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({url:document.getElementById('url').value, explain:document.getElementById('explain').checked})});
    const data=await response.json();
    if(!response.ok) throw new Error(data.error || 'Request failed');
    const score=Number(data.phishing_score);
    const status=document.getElementById('status');
    status.textContent=data.decision;
    status.className='status '+(data.decision==='LIKELY PHISHING'?'danger':data.decision==='LIKELY LEGITIMATE'?'good':'caution');
    document.getElementById('prob').textContent=(100*score).toFixed(1)+'%';
    document.getElementById('fill').style.width=(100*score).toFixed(1)+'%';
    document.getElementById('warning').textContent=data.warning;
    const list=document.getElementById('explain-list');list.replaceChildren();
    if(Array.isArray(data.explanation)){
      const heading=document.createElement('h2');heading.textContent='Top SHAP feature contributions';list.appendChild(heading);
      const p=document.createElement('div');p.className='help';p.textContent='Positive contributions increase the phishing score. Negative contributions reduce it. These are model explanations, not proof of malicious intent.';list.appendChild(p);
      for(const item of data.explanation){
        const row=document.createElement('div');row.className='feature';
        const name=document.createElement('span');name.textContent=item.feature+' ('+item.value+')';
        const impact=document.createElement('span');impact.className='impact '+(item.impact_on_phishing_score>0?'danger':'good');impact.textContent=(item.impact_on_phishing_score>=0?'+':'')+item.impact_on_phishing_score.toFixed(4);
        row.append(name,impact);list.appendChild(row);
      }
    }
    result.classList.add('visible');
  }catch(e){error.textContent=e.message || 'Error';}
  finally {button.disabled=false;button.textContent='Analyze URL';}
});
</script>
</body>
</html>

In [ ]:
%%writefile requirements.txt
# Python 3.10+ recommended. Install in a fresh virtual environment.
numpy>=1.26,<3
pandas>=2.0,<3
scikit-learn>=1.4,<2
xgboost>=2.0,<4
shap>=0.45,<1
tldextract>=5,<6
ucimlrepo>=0.0.7
matplotlib>=3.8,<4
seaborn>=0.13,<1
flask>=3,<4
joblib>=1.3,<2
nbformat>=5,<6

In [ ]:
%%writefile README.md
# URL-only phishing detection — reproducible academic lab

**Important:** PhiUSIIL's legitimate URL examples have severe path/`www.` distribution bias. The notebook deliberately *does not use* path, query, scheme, or `www.` as features. That makes the classifier robust to superficial changes in these components but unable to catch phishing that resides solely on the path of a benign-looking hostname. **No classifier trained on PhiUSIIL alone should be described as real-world reliable without an independent, representative benign-and-phishing benchmark.**

## Colab workflow

1. Open `phishing_detection_lab.ipynb` in Google Colab.
2. Runtime → Run all. The first cell installs all required libraries. Internet is needed to fetch UCI 967.
3. The notebook: audits data → removes duplicate URLs → domain-disjoint train/validation/test → builds the exact features defined in `features.py` → training-only feature selection → fits Logistic Regression, Random Forest, XGBoost and soft-voting → selects a **provisional** champion by validation F1 under an FPR constraint → plots ROC/PR curves → reports untouched test-set metrics → produces SHAP explanations and counterfactual invariance checks.
4. Final cell creates `model_bundle.joblib` and packages `phishing_flask_project.zip` (contains Flask app, frontend, feature extraction and model). Download that archive from Colab.
5. Optionally place a fresh, independently verified `external_urls.csv` next to the notebook with columns `URL,label` (0=phishing, 1=legitimate). The optional notebook cell evaluates it without fitting on it. **Do not reuse the old 145-URL phishing sample as an untouched final test.**

## Flask (local)

Unzip `phishing_flask_project.zip`, then:

```bash
python -m venv .venv
source .venv/bin/activate         # Windows: .venv\Scripts\activate
pip install -r requirements.txt
python app.py
```

Open http://127.0.0.1:5000/.

For POST integration:

```bash
curl -X POST http://127.0.0.1:5000/api/predict \
  -H 'Content-Type: application/json' \
  -d '{"url":"https://example.org/login","explain":true}'
```

Example JSON fields: `decision` (`LIKELY PHISHING`, `INSPECT`, or `LIKELY LEGITIMATE`), `predicted_label` (`0`, `null`, or `1`), `phishing_score`, and optional `explanation` (model-agnostic SHAP impact). **The score is not validated as a real-world probability.**

The server never follows or visits submitted URLs. It rejects empty/non-HTTP URLs, caps input size, uses a fixed feature schema, and binds to localhost by default. This is a demo, not a hardened public service. Add authentication, rate limits, HTTPS, monitoring, and proper deployment configuration before exposing it publicly. SHAP is computationally expensive and should be rate-limited in a public app.

## Feature policy

`features.py` is the **single source of truth** used during training, testing, SHAP and Flask inference. It uses only hostname and registered-domain string metrics. `URLSimilarityIndex`, HTML features, path/query lengths and `www.` are excluded on purpose. `tldextract` uses its bundled PSL snapshot offline for consistent features; update dependencies cautiously and retrain if definitions change.

## Model selection & limitations

- 0 = phishing, 1 = legitimate throughout. ROC-AUC and PR-AUC use `(label==0)` as the positive class, and `predict_proba` is indexed through the model's `classes_`.
- Only the development split is used for mutual information and correlation pruning. The validation split chooses a decision threshold and model; the test split is touched **once** for final performance reporting.
- Groups are registered domains, to reduce leakage between related URLs. The splits are not guaranteed to have identical class proportions.
- Validation target: a false-alarm rate no greater than 5%, then maximize phishing F1. This threshold is only a laboratory operating point, not a real-world guarantee.
- A low-risk answer is **not** verification that a site is safe; lexical-only detection misses many phishing websites, particularly short simple hostnames and attacks on compromised legitimate domains.

## Basic tests

```bash
python -m unittest discover -s tests
```

In [ ]:
%%writefile tests/test_features.py
import unittest
import pandas as pd
from features import FEATURE_NAMES, extract_lexical_features, registered_domain_from_url


class TestFeatures(unittest.TestCase):
    def test_schema_and_numeric(self):
        record = extract_lexical_features('https://blog.example.org/posts?id=2')
        self.assertEqual(list(record), FEATURE_NAMES)
        self.assertTrue(all(isinstance(v, (float, int)) for v in record.values()))

    def test_www_and_path_are_invariant(self):
        base = extract_lexical_features('https://www.example.com/')
        changed = extract_lexical_features('http://example.com/docs?a=1#foo')
        self.assertEqual(base, changed)

    def test_labelled_domain_group(self):
        self.assertEqual(registered_domain_from_url('https://a.example.co.uk/path'),
                         registered_domain_from_url('http://b.example.co.uk/'))

    def test_reject_non_http_scheme(self):
        with self.assertRaises(ValueError):
            extract_lexical_features('javascript:alert(1)')

    def test_ip(self):
        self.assertEqual(extract_lexical_features('http://192.0.2.7/a')['host_is_ip'], 1)


if __name__ == '__main__':
    unittest.main()

### Package downloadable Flask project and optionally run a local API smoke test
The generated ZIP is ready to extract beside your local `python app.py`. Flask runs only on localhost. The frontend calls `/api/predict` with POST JSON.

In [ ]:
import zipfile
files_to_zip=['model_bundle.joblib','features.py','app.py','templates/index.html',
              'requirements.txt','README.md','tests/test_features.py']
with zipfile.ZipFile('phishing_flask_project.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for item in files_to_zip:
        zf.write(item,arcname=item)
print('Created:',Path('phishing_flask_project.zip').resolve())

# Flask smoke test without starting a public listener
from app import app
client=app.test_client()
response=client.post('/api/predict',json={'url':'https://www.example.org/docs?p=42','explain':False})
print('Flask /api/predict status:',response.status_code)
print('Response:',response.get_json())
assert response.status_code==200

try:
    from google.colab import files
    files.download('phishing_flask_project.zip')
except ImportError:
    print('Not in Colab; ZIP file is in the current directory.')

## Lab conclusions and limitations

- Comparing models on a domain-disjoint split is **stronger than a random URL split**, but cannot remove source/collection bias.
- Training only on PhiUSIIL cannot validate generalization to modern short phishing hostnames, abused trusted domains, or benign URL paths/query parameters.
- By intentionally removing path/query, scheme and leading `www.` signals, the model will not change predictions for those superficial modifications; **this is a deliberate limitation**.
- The validation FPR ceiling and SHAP contributions explain only this dataset and model. They are not real-world safety guarantees.
- Use a *new*, independently labeled, representative external benchmark with both benign and phishing URLs before trusting/blocking based on output.
- Do not present the provisional highest-scoring model as objectively "best" for production. Evaluate external precision, recall, false-alarm rate, and domain shift first.